In [ ]:
# # Formatting notebook to fit the browser size
# # commented out so not to produce error

# from IPython.core.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
# Import data

# path of input file(s) - note that the file(s) has to be in a folder called data_inputs
input_data_file = os.path.join('data_inputs', filename) #note - this uses a variable called filename as the input file, if you are only running one file you can change this to the name of your file in quotation marks 
input_data_file = os.path.join('data_inputs', filename) #note - this uses a variable called filename as the input file, if you are only running one file you can change this to the name of your file in quotation marks 


#getting the base name of the file which will be used to name the output file in the same way
name = ((os.path.splitext(input_data_file))[0]).lstrip('data_inputs\\') #This takes the file name, splits it at the extension (e.g. csv), then takes the first element of this and removes the data_inputs\ folder location from it, this results in the file name

# define column headings for the dataframe
headings_list = ['Bin_number','Bin_diameter_lower', 'Diff_volume', 'Diff_number_perc', 'Diff_number_gram', 'Diff_surface_area', 'Diff_number']

# read in dataframe using pandas library 
input_data_df = pd.read_csv(input_data_file, sep='\,', engine='python', skiprows=55,
                            names=headings_list)

# calculate bin width in a new column
input_data_df['Bin_width'] = input_data_df['Bin_diameter_lower'].shift(-1) - input_data_df['Bin_diameter_lower']

# read off the final bin number into a variable, called input_data_bin_max, and then drop this row
input_data_bin_max = input_data_df.iloc[-1,1]
input_data_df = input_data_df.drop(input_data_df.index[input_data_df.tail(1).index])

# set 'Bin_number' column to be the index
input_data_df = input_data_df.set_index('Bin_number')

# change datatype of index from float to integer
input_data_df.index = input_data_df.index.astype('int')

# Calculate mid-point of bins
input_data_df['Bin_midpoint'] = input_data_df['Bin_diameter_lower']+input_data_df['Bin_width']/2

# Calc Diff_volume_density using: Diff_volume/Bin_width
input_data_df['Diff_volume_density'] = input_data_df['Diff_volume']/input_data_df['Bin_width']
Diff_volume_density = input_data_df['Diff_volume_density'].tolist()

# convert input data to lists
bin_midpoint_input_list = input_data_df['Bin_midpoint'].tolist()

In [ ]:
#Taking optional values which are fed into the Running script
#If not present in Running script they are set to default values

if 'x_value_to_plot_to' in globals():
      max_x_value = x_value_to_plot_to
else:
    max_x_value = 50
    
    
if 'display_initial_parameter_graphs' in globals():
    display_initial_parameter_graphs = display_initial_parameter_graphs
    PATH = 'output/Graphs_of_initial_parameters'
    if not os.path.exists(PATH):
            os.makedirs(PATH)
else:
    display_initial_parameter_graphs = 'FALSE'


if 'lognormal_normal_initial_gamma' in globals():
    gamma_ = lognormal_normal_initial_gamma
else:
    gamma_ = 0.2

if 'lognormal_normal_initial_mu_B_type_granule_curve' in globals():
    mu_1_ = lognormal_normal_initial_mu_B_type_granule_curve
else:
    mu_1_ = 1.5

if 'lognormal_normal_initial_sigma_B_type_granule_curve' in globals():
    sigma_1_ = lognormal_normal_initial_sigma_B_type_granule_curve
else:
    sigma_1_ = 0.5

if 'lognormal_normal_initial_mu_A_type_granule_curve' in globals():
    mu_2_ = lognormal_normal_initial_mu_A_type_granule_curve
else:
    mu_2_ = 20
    
if 'lognormal_normal_initial_sigma_A_type_granule_curve' in globals():
    sigma_2_ = lognormal_normal_initial_sigma_A_type_granule_curve
else:
    sigma_2_ = 5


if 'normal_normal_initial_gamma' in globals():
    gamma_2_ = normal_normal_initial_gamma
else:
    gamma_2_ = 0.2

if 'normal_normal_initial_mu_B_type_granule_curve' in globals():
    mu_3_ = normal_normal_initial_mu_B_type_granule_curve
else:
    mu_3_ = 5

if 'normal_normal_initial_sigma_B_type_granule_curve' in globals():
    sigma_3_ = normal_normal_initial_sigma_B_type_granule_curve
else:
    sigma_3_ =2

if 'normal_normal_initial_mu_A_type_granule_curve' in globals():
    mu_4_ = normal_normal_initial_mu_A_type_granule_curve
else:
    mu_4_ = 20
    
if 'normal_normal_initial_sigma_A_type_granule_curve' in globals():
    sigma_4_ = normal_normal_initial_sigma_A_type_granule_curve
else:
    sigma_4_ = 5  
    
    

if 'lognormal_lognormal_initial_gamma' in globals():
    gamma_3_ = lognormal_lognormal_initial_gamma
else:
    gamma_3_ = 0.5

if 'lognormal_lognormal_initial_mu_B_type_granule_curve' in globals():
    mu_5_ = lognormal_lognormal_initial_mu_B_type_granule_curve
else:
    mu_5_ = 2

if 'lognormal_lognormal_initial_sigma_B_type_granule_curve' in globals():
    sigma_5_ = lognormal_lognormal_initial_sigma_B_type_granule_curve
else:
    sigma_5_ = 0.5

if 'lognormal_lognormal_initial_mu_A_type_granule_curve' in globals():
    mu_6_ = lognormal_lognormal_initial_mu_A_type_granule_curve
else:
    mu_6_ = 3
    
if 'lognormal_lognormal_initial_sigma_A_type_granule_curve' in globals():
    sigma_6_ = lognormal_lognormal_initial_sigma_A_type_granule_curve
else:
    sigma_6_ = 0.2

    

In [ ]:
# Define the lognormal and normal functions

def lognormal_func(x_, mu_1_, sigma_1_):
    return (  (1/(x_*sigma_1_*np.sqrt(2.*np.pi)))*np.exp( -((np.log(x_)-mu_1_)/sigma_1_)**2/2. )  )

def normal_func(x_, mu_2_, sigma_2_):
    return (  (1/(sigma_2_*np.sqrt(2.*np.pi)))*np.exp( -( (x_ - mu_2_)/sigma_2_ )**2/2. )  )


In [ ]:
## LOGNORMAL-NORMAL FUNCTION

In [ ]:
# Define the lognormal-normal function

A = 100 #A is the total area under curve (100), so it is a defined value and doesn't need to be determined during the fit

def lognormal_normal_func(x_, gamma_, mu_1_, sigma_1_, mu_2_, sigma_2_): 
    return ( A*((gamma_*lognormal_func(x_, mu_1_, sigma_1_)) + ((1-gamma_)*normal_func(x_, mu_2_, sigma_2_))) )

In [ ]:
# Define a function to fit the lognogrmal-normal function

#initial parameter values:
        #default values
        # gamma_ = 0.2
        # mu_1_ = 1.5
        # sigma_1_ = 0.5
        # mu_2_ = 20
        # sigma_2_ = 5

initial_params_ = [gamma_, mu_1_, sigma_1_, mu_2_, sigma_2_]

def func_lognormal_normal_fit(bin_midpoint_input_, Diff_volume_density):
    #initial parameter values:  
    fit_params_, fit_cov_ = scipy.optimize.curve_fit(lognormal_normal_func,bin_midpoint_input_, Diff_volume_density, 
                                                    p0 = initial_params_, 
                                                     bounds = ([0,0,0,0,0], [1,np.inf,np.inf,np.inf,np.inf])) #note that the bounds of gamma have been fixed so that it can only take values from 0 to 1, everything else can take values from 0 to infinity
    fit_params_err_ = np.sqrt(np.diag(fit_cov_))
    return fit_params_, fit_params_err_

In [ ]:
# Run the lognormal-normal fitting

#create empty lists which the successful and non-successful fittings will be stored in
failed_lognormal_normal = []
successful_lognormal_normal = []
fitting_failed_lognormal_normal = []

#run the lognormal-normal fit with the exception catcher
try:
    output_lognormal_normal_fit_params_list, output_lognormal_normal_fit_err = func_lognormal_normal_fit(bin_midpoint_input_list, Diff_volume_density)

except RuntimeError:
        failed_lognormal_normal = name.split('\n') #we need the \n if not we get \n included in the sample names when we make a list of samples in batch mode
        
        #this sets the parameters to fitting failed so in the output file we get this message
        B_granule_diameter_lognormal_normal = "fitting failed"
        A_granule_diameter_lognormal_normal = "fitting failed"
        B_granule_area_lognormal_normal = "fitting failed"
        A_granule_area_lognormal_normal = "fitting failed"
        B_granule_content_lognormal_normal = "fitting failed"
        lognormal_normal_uncertainity = "fitting failed"
        lognormal_normal_standard_error_of_regression = "fitting failed"
    
        #this is used in an if statement during plotting so that if failed we don't get a graph
        fitting_failed_lognormal_normal = "yes"

else:
    successful_lognormal_normal = name.split('\n') #we need the \n if not we get \n included in the sample names when we make a list of samples in batch mode
    
    #Output parameters - the fitting produces a dataframe called output_lognormal_normal_fit_params_list which has the following format [gamma_, mu_1_, sigma_1_, mu_2_, sigma_2_], where 1 refers to lognormal curve for B-type granules and 2 refers to normal curve for A-type granules
    gamma_lognormal_normal_optimised = output_lognormal_normal_fit_params_list[0]
    mu_1_lognormal_normal_optimised = output_lognormal_normal_fit_params_list[1]
    sigma_1_lognormal_normal_optimised = output_lognormal_normal_fit_params_list[2]
    mu_2_lognormal_normal_optimised = output_lognormal_normal_fit_params_list[3]
    sigma_2_lognormal_normal_optimised = output_lognormal_normal_fit_params_list[4]

    #Getting the parameters from the model - note that the B-type granule curve is lognormal and the A-type granule curve is normal
    
    #lognorm mean is equal to exp(mu + (sigma^2/2))
    B_granule_diameter_lognormal_normal = np.exp(mu_1_lognormal_normal_optimised + ((sigma_1_lognormal_normal_optimised**2)/2))
    
    #norm mean is equal to mu
    A_granule_diameter_lognormal_normal = mu_2_lognormal_normal_optimised
    
    #lognorm variance is equal to [exp(sigma^2)-1] exp(2*mu + sigma^2)
    B_granule_variance_lognormal_normal = ((np.exp(sigma_1_lognormal_normal_optimised**2)-1)*(np.exp(2*mu_1_lognormal_normal_optimised+sigma_1_lognormal_normal_optimised**2)))

    #normal variance is equal to sigma^2
    A_granule_variance_lognormal_normal = sigma_2_lognormal_normal_optimised**2
    
   
    #granule content
    B_granule_content_lognormal_normal = 100*gamma_lognormal_normal_optimised
    A_granule_content_lognormal_normal = 100*(1-gamma_lognormal_normal_optimised)

    #uncertainty for each parameter - the fitting produces a dataframe of uncertainties called output_lognormal_normal_fit_err which has the following format [gamma_, mu_1_, sigma_1_, mu_2_, sigma_2_], where 1 refers to lognormal curve for B-type granules and 2 refers to normal curve for A-type granules
    gamma_lognormal_normal_uncertainty = output_lognormal_normal_fit_err[0]
    mu_1_lognormal_normal_uncertainty = output_lognormal_normal_fit_err[1]
    sigma_1_lognormal_normal_uncertainty = output_lognormal_normal_fit_err[2]
    mu_2_lognormal_normal_uncertainty = output_lognormal_normal_fit_err[3]
    sigma_2_lognormal_normal_uncertainty = output_lognormal_normal_fit_err[4]    
    
    #total uncertainity
    lognormal_normal_uncertainity = np.sum(output_lognormal_normal_fit_err)
    
    #standard error of regression
    lognormal_normal_vals_list = [lognormal_normal_func(input_data_df['Bin_midpoint'].tolist()[i_], *output_lognormal_normal_fit_params_list) #the star takes all the values from output_fit_params and puts it in here
                        for i_ in range(len(input_data_df['Diff_volume_density']))]
    lognormal_normal_standard_error_of_regression = np.sqrt( (1/len(input_data_df['Diff_volume_density']-2)) * (sum((np.array(Diff_volume_density) - np.array(lognormal_normal_vals_list))**2) / sum((np.array( np.array(input_data_df['Bin_midpoint'].tolist()) - mean(input_data_df['Bin_midpoint'].tolist()) ))**2)) )

In [ ]:
#an alternative way to calculate the B-type granule content is to integrate under the curve - this is the code for this, gives same output as using gamma


#  #areas under curves - with the scipy integrate function: the individual functions have been redefined using the output parameters from the fit 
#     #and the integrate function is used, the output is a tuple with an estimated value of the integral first and the second value is an upper bound on the error
#     #B granules - lognorm fitting
#     def lognorm_func_B_granules(x_): 
#         # (  (1/(x_*sigma_1_*np.sqrt(2.*np.pi)))*np.exp( -((np.log(x_)-mu_1_)/sigma_1_)**2/2. )  )
#         return (  gamma_lognormal_normal_optimised*(1/(x_*sigma_1_lognormal_normal_optimised*np.sqrt(2.*np.pi)))*np.exp( -((np.log(x_)-mu_1_lognormal_normal_optimised)/sigma_1_lognormal_normal_optimised)**2/2. )  )    
#     B_granule_area_lognormal_normal = scipy.integrate.quad(lognorm_func_B_granules,0, np.inf) #integrate between 0 and infinity
    
#     #A granules - normal fitting
#     def normal_func_A_granules(x_): 
#         #(1/(sigma_2_*np.sqrt(2.*np.pi)))*np.exp( -( (x_ - mu_2_)/sigma_2_ )**2/2. ) 
#         return (  (1-gamma_lognormal_normal_optimised)*(1/(sigma_2_lognormal_normal_optimised*np.sqrt(2.*np.pi)))*np.exp( -( (x_ - mu_2_lognormal_normal_optimised)/sigma_2_lognormal_normal_optimised)**2/2. )  )  
#     A_granule_area_lognormal_normal = scipy.integrate.quad(normal_func_A_granules, 0, np.inf) #integrate between 0 and infinity
   
#    #Total_area_lognormal_normal = A_granule_area_lognormal_normal[0] + B_granule_area_lognormal_normal[0]
#     B_granule_content_lognormal_normal = 100*B_granule_area_lognormal_normal[0] #the first element of B_granule_area_lognormal_normal is the area under the curve
#     A_granule_content_lognormal_normal = 100 - B_granule_content_lognormal_normal
   

In [ ]:
# DOUBLE NORMAL FUNCTION

In [ ]:
#Define the double normal function

A = 100 #A is the total area under curve (100), so it is a defined value and doesn't need to be determined during the fit

def double_normal_func(x_, gamma_2_, mu_3_, sigma_3_, mu_4_, sigma_4_):
        return ( A* ( (gamma_2_*((1/(sigma_3_*np.sqrt(2.*np.pi)))*np.exp( -( (x_ - mu_3_)/sigma_3_ )**2/2. ))) + ((1-gamma_2_)*(1/(sigma_4_*np.sqrt(2.*np.pi)))*np.exp( -( (x_ - mu_4_)/sigma_4_ )**2/2. ))) )

In [ ]:
# Define a function to fit the double normal function

#initial parameter values:
        # default values
        # gamma_2_ = 0.2
        # mu_3_ = 5
        # sigma_3_ = 2
        # mu_4_ = 20
        # sigma_4_ = 5

initial_params_double_norm = [gamma_2_, mu_3_, sigma_3_, mu_4_, sigma_4_]



def func_double_normal_fit(bin_midpoint_input_, Diff_volume_density):
    fit_params_double_normal, fit_cov_double_normal = scipy.optimize.curve_fit(double_normal_func,bin_midpoint_input_, Diff_volume_density, 
                                                    p0 = initial_params_double_norm, 
                                                    bounds = ([0,0,0,0,0], [1,np.inf,np.inf,np.inf,np.inf])) #note that the bounds of gamma have been fixed so that it can only take values from 0 to 1, everything else can take values from 0 to infinity
              
    fit_params_err_double_normal = np.sqrt(np.diag(fit_cov_double_normal))
    return fit_params_double_normal, fit_params_err_double_normal

#This function will return: {list of fit parameters}, {list of fit uncertanties}


In [ ]:
# Run the double normal fitting

#create empty lists which I will store failed/successful fittings in
failed_double_normal = []
successful_double_normal = []
fitting_failed_double_normal = []


#run the double normal fit with the exception catcher
try:
    output_double_normal_fit_params_list, output_double_normal_fit_err = func_double_normal_fit(bin_midpoint_input_list, Diff_volume_density)

except RuntimeError:
    failed_double_normal = name.split('\n') #we need the \n if not we get \n included in the sample names when we make a list of samples in batch mode

    #this sets the parameters to fitting failed so in the output file we get this message
    B_granule_diameter_double_normal = "fitting failed"
    A_granule_diameter_double_normal = "fitting failed"
    B_granule_area_double_normal = "fitting failed"
    A_granule_area_double_normal = "fitting failed"
    B_granule_content_double_normal = "fitting failed"
    double_normal_uncertainity = "fitting failed"
    double_normal_standard_error_of_regression = "fitting failed"
    
    #this is used in an if statement during plotting so that if failed we don't get a graph
    fitting_failed_double_normal = "yes"

else:
    successful_double_normal = name.split('\n') #we need the \n if not we get \n included in the sample names when we make a list of samples in batch mode
    
    #Output parameters - the fitting produces a dataframe called output_lognormal_normal_fit_params_list which has the following format [gamma_2_, mu_3_, sigma_3_, mu_4_, sigma_4_], where 2 refers to normal curve for B-type granules and 4 refers to normal curve for A-type granules
    gamma_2_double_normal_optimised = output_double_normal_fit_params_list[0]
    mu_3_double_normal_optimised = output_double_normal_fit_params_list[1]
    sigma_3_double_normal_optimised = output_double_normal_fit_params_list[2]
    mu_4_double_normal_optimised = output_double_normal_fit_params_list[3]
    sigma_4_double_normal_optimised = output_double_normal_fit_params_list[4]
    
    

    #Getting the parameters from the model
    
    #norm mean is equal to mu
    B_granule_diameter_double_normal = mu_3_double_normal_optimised
    A_granule_diameter_double_normal = mu_4_double_normal_optimised
    
   
    #normal variance is equal to sigma^2
    B_granule_variance_double_normal = sigma_3_double_normal_optimised**2
    A_granule_variance_double_normal = sigma_4_double_normal_optimised**2
    
    
   #Granule contents
    B_granule_content_double_normal = 100*gamma_2_double_normal_optimised
    A_granule_content_double_normal = 100*(1-gamma_2_double_normal_optimised)

    #uncertainty for each parameter - the fitting produces a dataframe of uncertainties called output_double_normal_fit_err which has the following format [gamma_2_, mu_3_, sigma_3_, mu_4_, sigma_4_], where 1 refers to lognormal curve for B-type granules and 2 refers to normal curve for A-type granules
    gamma_2_double_normal_uncertainty = output_double_normal_fit_err[0]
    mu_3_double_normal_uncertainty = output_double_normal_fit_err[1]
    sigma_3_double_normal_uncertainty = output_double_normal_fit_err[2]
    mu_4_double_normal_uncertainty = output_double_normal_fit_err[3]
    sigma_4_double_normal_uncertainty = output_double_normal_fit_err[4]    
    
    #total uncertainity
    double_normal_uncertainity = np.sum(output_double_normal_fit_err)
    
    #standard error of regression
    double_norm_vals_list = [double_normal_func(input_data_df['Bin_midpoint'].tolist()[i_], *output_double_normal_fit_params_list) #the star takes all the values from output_fit_params and puts it in here
                        for i_ in range(len(input_data_df['Diff_volume_density']))]
    double_normal_standard_error_of_regression = np.sqrt( (1/len(input_data_df['Diff_volume_density']-2)) * (sum((np.array(input_data_df['Diff_volume_density'].tolist()) - np.array(double_norm_vals_list))**2) / sum((np.array( np.array(input_data_df['Bin_midpoint'].tolist()) - mean(input_data_df['Bin_midpoint'].tolist()) ))**2)) )
    

In [ ]:
# DOUBLE LOGNORMAL FUNCTION

In [ ]:
# Define the double lognormal function

A = 100 #A is the total area under curve (100), so it is a defined value and doesn't need to be determined during the fit

def double_lognormal_func(x_, gamma_3_, mu_5_, sigma_5_, mu_6_, sigma_6_):
     return (A*(  (gamma_3_*(1/(x_*sigma_5_*np.sqrt(2.*np.pi)))*np.exp( -((np.log(x_)-mu_5_)/sigma_5_)**2/2. )) + 
         ((1-gamma_3_)*(1/(x_*sigma_6_*np.sqrt(2.*np.pi)))*np.exp( -((np.log(x_)-mu_6_)/sigma_6_)**2/2. ))  ))

In [ ]:
# Define a function to fit the the double lognromal function

#initial parameter values:
        #default values
        # gamma_3_ = 0.5
        # mu_5_ = 2
        # sigma_5_ = 0.5
        # mu_6_ = 3
        # sigma_6_ = 0.2

initial_params_double_lognormal = [gamma_3_, mu_5_, sigma_5_, mu_6_, sigma_6_]


def func_double_lognormal_fit(bin_midpoint_input_, Diff_volume_density):
    fit_params_double_lognormal, fit_cov_double_lognormal = scipy.optimize.curve_fit(double_lognormal_func,bin_midpoint_input_, Diff_volume_density, 
                                                    p0 = initial_params_double_lognormal,
                                                    bounds = ([0,0,0,0,0], [1,np.inf,np.inf,np.inf,np.inf])) #note that the bounds of gamma have been fixed so that it can only take values from 0 to 1, everything else can take values from 0 to infinity

    
    fit_params_err_double_lognormal = np.sqrt(np.diag(fit_cov_double_lognormal))
    return fit_params_double_lognormal, fit_params_err_double_lognormal

#This function will return: {list of fit parameters}, {list of fit uncertanties}

In [ ]:
# Run the double lognormal fitting

#create empty lists which I will store failed/successful fittings in
failed_double_lognormal = []
successful_double_lognormal = []
fitting_failed_double_lognormal = []


#run the fit with the exception catcher
try:
    output_double_lognormal_fit_params_list, output_double_lognormal_fit_err = func_double_lognormal_fit(bin_midpoint_input_list, Diff_volume_density)

except RuntimeError:
    failed_double_lognormal = name.split('\n') #we need the \n if not we get \n included in the sample names when we make a list of samples in batch mode
    
    #this sets the parameters to fitting failed so in the output file we get this message
    B_granule_diameter_double_lognormal = "fitting failed"
    A_granule_diameter_double_lognormal = "fitting failed"
    B_granule_area_double_lognormal = "fitting failed"
    A_granule_area_double_lognormal = "fitting failed"
    B_granule_content_double_lognormal = "fitting failed"
    double_lognormal_uncertainity = "fitting failed"
    double_lognormal_standard_error_of_regression = "fitting failed"
    
    #this is used in an if statement during plotting so that if failed we don't get a graph
    fitting_failed_double_lognormal = "yes"

else:
    successful_double_lognormal = name.split('\n') #we need the \n if not we get \n included in the sample names when we make a list of samples in batch mode
    #Output parameters - the fitting produces a dataframe called output_lognormal_normal_fit_params_list which has the following format [gamma_3_, mu_5_, sigma_5_, mu_6_, sigma_6_], where 5 refers to lognormal curve for B-type granules and 6 refers to lognormal curve for A-type granules
    gamma_3_double_lognormal_optimised = output_double_lognormal_fit_params_list[0]
    mu_5_double_lognormal_optimised = output_double_lognormal_fit_params_list[1]
    sigma_5_double_lognormal_optimised = output_double_lognormal_fit_params_list[2]
    mu_6_double_lognormal_optimised = output_double_lognormal_fit_params_list[3]
    sigma_6_double_lognormal_optimised = output_double_lognormal_fit_params_list[4]
    
    
    #Getting the parameters from the models
    #lognorm mean is equal to exp(mu + (sigma^2/2))
    B_granule_diameter_double_lognormal = np.exp(mu_5_double_lognormal_optimised + ((sigma_5_double_lognormal_optimised**2)/2))
    A_granule_diameter_double_lognormal = np.exp(mu_6_double_lognormal_optimised + ((sigma_6_double_lognormal_optimised**2)/2))
    
    
    
    #lognorm variance is equal to [exp(sigma^2)-1] exp(2*mu + sigma^2)
    B_granule_variance_double_lognormal = ((np.exp(sigma_5_double_lognormal_optimised**2)-1)*(np.exp(2*mu_5_double_lognormal_optimised+sigma_5_double_lognormal_optimised**2)))
    A_granule_variance_double_lognormal = ((np.exp(sigma_6_double_lognormal_optimised**2)-1)*(np.exp(2*mu_6_double_lognormal_optimised+sigma_6_double_lognormal_optimised**2)))

       
   #Granule contents
    B_granule_content_double_lognormal = 100*gamma_3_double_lognormal_optimised
    A_granule_content_double_lognormal = 100*(1-gamma_3_double_lognormal_optimised)

    #uncertainty for each parameter - the fitting produces a dataframe of uncertainties called output_double_lognormal_fit_err which has the following format [gamma_3_, mu_5_, sigma_5_, mu_6_, sigma_6_], where 1 refers to lognormal curve for B-type granules and 2 refers to normal curve for A-type granules
    gamma_3_double_lognormal_uncertainty = output_double_lognormal_fit_err[0]
    mu_5_double_lognormal_uncertainty = output_double_lognormal_fit_err[1]
    sigma_5_double_lognormal_uncertainty = output_double_lognormal_fit_err[2]
    mu_6_double_lognormal_uncertainty = output_double_lognormal_fit_err[3]
    sigma_6_double_lognormal_uncertainty = output_double_lognormal_fit_err[4]    
    
    #total uncertainity
    double_lognormal_uncertainity = np.sum(output_double_lognormal_fit_err)
    
    #standard error of regression
    double_lognorm_vals_list = [double_lognormal_func(input_data_df['Bin_midpoint'].tolist()[i_], *output_double_lognormal_fit_params_list) #the star takes all the values from output_fit_params and puts it in here
                    for i_ in range(len(input_data_df['Diff_volume_density']))]
    double_lognormal_standard_error_of_regression = np.sqrt( (1/len(input_data_df['Diff_volume_density']-2)) * (sum((np.array(input_data_df['Diff_volume_density'].tolist()) - np.array(double_lognorm_vals_list))**2) / sum((np.array( np.array(input_data_df['Bin_midpoint'].tolist()) - mean(input_data_df['Bin_midpoint'].tolist()) ))**2)) )

In [ ]:
#PLOTTING

In [ ]:
#Plotting initial parameter - optional extra    
#Plots the initial parameters to visualise how close the initial parameters fit
#This can be useful if your curves are very different from normal and the initial parameters need to be adjusted
#Will only run if display_initial_parameter_graphs = TRUE

if (display_initial_parameter_graphs == 'TRUE'):
    
    #setting up plots
    fig, ax = plt.subplots(3,1,figsize=(15,25))
    fig.suptitle(name, fontsize=20, y=1.01,   fontname="Arial", weight="bold")
    fig.subplots_adjust(top=0.95)
    fig.tight_layout(h_pad=8)
    font = font_manager.FontProperties(family='Arial',
                                       style='normal', size=11)
    axfont = 11
    
    # define x_values to plot
    x_vals_plot = np.arange(0.5,max_x_value,0.5)
    
    
    
    

    # define curves to fit   
    double_normal_vals_curve = [double_normal_func(i_, initial_params_double_norm[0], initial_params_double_norm[1],initial_params_double_norm[2], 
                                           initial_params_double_norm[3],initial_params_double_norm[4]) 
                            for i_ in x_vals_plot]

    double_lognormal_vals_curve = [double_lognormal_func(i_, initial_params_double_lognormal[0],initial_params_double_lognormal[1],initial_params_double_lognormal[2], 
                                           initial_params_double_lognormal[3],initial_params_double_lognormal[4]) 
                            for i_ in x_vals_plot]
    
    lognormal_normal_vals_curve = [lognormal_normal_func(i_, initial_params_[0],initial_params_[1],initial_params_[2], initial_params_[3], initial_params_[4]) 
                            for i_ in x_vals_plot]
    

    
        # define curves to fit
    for spine in ['left','right','top','bottom']:
        ax[0].spines[spine].set_color('k')
    ax[0].set_facecolor('white')
    ax[0].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(),
            linewidth=2, color='k',label='Data' )
    ax[0].plot(x_vals_plot, double_normal_vals_curve,
            linewidth=2, color='b',label='N-N' )
    ax[0].plot(x_vals_plot, double_normal_vals_curve,
            linewidth=2, color='w', alpha=0, label=('gamma = %.2f\nmu B-type granule curve = %.2f\nsigma B-type granule curve = %.2f\nmu A-type granule curve = %.2f\nsigma A-type granule curve = %.2f ' ) % (initial_params_double_norm[0], initial_params_double_norm[1],initial_params_double_norm[2], 
                                           initial_params_double_norm[3],initial_params_double_norm[4])  )  
    ax[0].set_title('Initial parameters for Normal - Normal (N-N)', fontsize=16, fontname="Arial", weight="bold"),
    ax[0].set_xlabel("Diameter ($\mu $m)", fontsize=18)
    ax[0].xaxis.set_major_locator(plt.MaxNLocator(6))
    ax[0].set_ylabel("Volume Density (%)", fontsize=18)
    ax[0].tick_params(axis='both', which='major', labelsize=16)
    leg_00 = ax[0].legend(fontsize=16, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
    leg_00.get_frame().set_edgecolor('k')
    plt.tight_layout() 
    fig.subplots_adjust(top=0.9)
    ax[0].set_xlim(0,max_x_value)
    for tick in ax[0].get_xticklabels():
        tick.set_fontname("Arial")
    for tick in ax[0].get_yticklabels():
        tick.set_fontname("Arial")
    ax[0].xaxis.label.set_color('black')
    ax[0].yaxis.label.set_color('black')
    ax[0].tick_params(axis='both', which='major', colors='black')
    for item in ([ax[0].xaxis.label, ax[0].yaxis.label] +
             ax[0].get_xticklabels() + ax[0].get_yticklabels()):
        item.set_fontsize(axfont)
    ax[0].set_ylim(bottom=0)
    ax[0].set_xlim(left=0)

    
    
    for spine in ['left','right','top','bottom']:
        ax[1].spines[spine].set_color('k')
    ax[1].set_facecolor('white')
    ax[1].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(),
            linewidth=2, color='k',label='Data' )
    ax[1].plot(x_vals_plot, double_lognormal_vals_curve,
            linewidth=2, color='b',label='L-L' )
    ax[1].set_title('Initial parameters for Log-normal - Log-normal (L-L)', fontsize=16, fontname="Arial", weight="bold")
    ax[1].plot(x_vals_plot, double_lognormal_vals_curve,
            linewidth=2, color='w', alpha=0, label=('gamma = %.2f\nmu B-type granule curve = %.2f\nsigma B-type granule curve = %.2f\nmu A-type granule curve = %.2f\nsigma A-type granule curve = %.2f ' ) % (initial_params_double_lognormal[0],initial_params_double_lognormal[1],initial_params_double_lognormal[2], 
                                           initial_params_double_lognormal[3],initial_params_double_lognormal[4])  )  
    ax[1].set_xlabel("Diameter ($\mu $m)", fontsize=18)
    ax[1].xaxis.set_major_locator(plt.MaxNLocator(6))
    ax[1].set_ylabel("Volume Density (%)", fontsize=18)
    ax[1].tick_params(axis='both', which='major', labelsize=16)
    leg_00 = ax[1].legend(fontsize=16, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
    leg_00.get_frame().set_edgecolor('k')
    plt.tight_layout()
    ax[1].set_xlim(0,max_x_value)
    for tick in ax[1].get_xticklabels():
        tick.set_fontname("Arial")
    for tick in ax[1].get_yticklabels():
        tick.set_fontname("Arial")
    ax[1].xaxis.label.set_color('black')
    ax[1].yaxis.label.set_color('black')
    ax[1].tick_params(axis='both', which='major', colors='black')
    for item in ([ax[1].xaxis.label, ax[1].yaxis.label] +
             ax[1].get_xticklabels() + ax[1].get_yticklabels()):
        item.set_fontsize(axfont)
    ax[1].set_ylim(bottom=0)
    ax[1].set_xlim(left=0)
    
    
    for spine in ['left','right','top','bottom']:
        ax[2].spines[spine].set_color('k')
    ax[2].set_facecolor('white')
    ax[2].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(), linewidth=2, color='k',label='Data' )
    ax[2].plot(x_vals_plot, lognormal_normal_vals_curve,linewidth=2, color='b',label='L-N' )
    ax[2].set_title('Initial parameters for Log-normal - Normal (L-N)', fontsize=16, fontname="Arial", weight="bold")
    ax[2].plot(x_vals_plot, lognormal_normal_vals_curve,
            linewidth=2, color='w', alpha=0,  label=('gamma = %.2f\nmu B-type granule curve = %.2f\nsigma B-type granule curve = %.2f\nmu A-type granule curve = %.2f\nsigma A-type granule curve = %.2f ' ) % (initial_params_[0],initial_params_[1],initial_params_[2], initial_params_[3], initial_params_[4])  ) 
    ax[2].set_xlabel("Diameter ($\mu $m)", fontsize=18)
    ax[2].xaxis.set_major_locator(plt.MaxNLocator(6))
    ax[2].set_ylabel("Volume Density (%)", fontsize=18)
    ax[2].tick_params(axis='both', which='major', labelsize=16)
    leg_00 = ax[2].legend(fontsize=16, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
    leg_00.get_frame().set_edgecolor('k')
    plt.tight_layout() 
    fig.subplots_adjust(top=0.9)
    ax[2].set_xlim(0,max_x_value)
    for tick in ax[2].get_xticklabels():
        tick.set_fontname("Arial")
    for tick in ax[2].get_yticklabels():
        tick.set_fontname("Arial")
    ax[2].xaxis.label.set_color('black')
    ax[2].yaxis.label.set_color('black')
    ax[2].tick_params(axis='both', which='major', colors='black')
    for item in ([ax[2].xaxis.label, ax[2].yaxis.label] +
             ax[2].get_xticklabels() + ax[2].get_yticklabels()):
        item.set_fontsize(axfont)
    ax[2].set_ylim(bottom=0)
    ax[2].set_xlim(left=0)
    
    #Saving plots in a folder called output
    plot_name = (name.strip('\n')+"_output.pdf")
    fig.savefig('output/Graphs_of_initial_parameters/'+name+"initial_parameters.pdf", bbox_inches = 'tight', dpi=300)


In [ ]:
#Getting values for all graphs to plot

#x values
x_vals_plot = np.arange(0.5,max_x_value,0.1)

    
#Normal - normal
double_normal_vals_curve = [double_normal_func(i_, *output_double_normal_fit_params_list) 
                         for i_ in x_vals_plot]
#B-type granule graph
# normal is defined as: (  (1/(sigma*np.sqrt(2.*np.pi)))*np.exp( -( (x_ - mu)/sigma )**2/2. )  )
double_normal_B_vals_curve = [normal_func(i_, mu_3_double_normal_optimised, sigma_3_double_normal_optimised) for i_ in x_vals_plot] #don't plot this - not adjusted for A and gamma
double_normal_B_vals_curve_to_plot = [i * 100 * gamma_2_double_normal_optimised for i in double_normal_B_vals_curve] #need this step to adjust curve for the value of A and gamma

#A-type granule graph
# normal is defined as: (  (1/(sigma_*np.sqrt(2.*np.pi)))*np.exp( -( (x_ - mu_)/sigma_ )**2/2. )  )
double_normal_A_vals_curve = [normal_func(i_,mu_4_double_normal_optimised, sigma_4_double_normal_optimised ) for i_ in x_vals_plot] #don't plot this - not adjusted for A and gamma
double_normal_A_vals_curve_to_plot = [i * 100 * (1-gamma_2_double_normal_optimised) for i in double_normal_A_vals_curve] #need this step to adjust curve for the value of A and gamma





#lognormal-lognormal
double_lognormal_vals_curve = [double_lognormal_func(i_, *output_double_lognormal_fit_params_list) 
                         for i_ in x_vals_plot]
#B-type granule graph
# lognormal is defined as (1/(x_*sigma_*np.sqrt(2.*np.pi)))*np.exp( -((np.log(x_)-mu_)/sigma_)**2/2. )
double_lognormal_B_vals_curve = [lognormal_func(i_, mu_5_double_lognormal_optimised, sigma_5_double_lognormal_optimised) for i_ in x_vals_plot] #don't plot this - not adjusted for A and gamma
double_lognormal_B_vals_curve_to_plot = [i * 100 * gamma_3_double_lognormal_optimised for i in double_lognormal_B_vals_curve] #need this step to adjust curve for the value of A and gamma

#A-type granule graph
# lognormal is defined as (1/(x_*sigma_*np.sqrt(2.*np.pi)))*np.exp( -((np.log(x_)-mu_)/sigma_)**2/2. )
double_lognormal_A_vals_curve = [lognormal_func(i_, mu_6_double_lognormal_optimised, sigma_6_double_lognormal_optimised) for i_ in x_vals_plot] #don't plot this - not adjusted for A and gamma
double_lognormal_A_vals_curve_to_plot = [i * 100 * (1-gamma_3_double_lognormal_optimised) for i in double_lognormal_A_vals_curve] #need this step to adjust curve for the value of A and gamma





#lognormal-normal
lognormal_normal_vals_curve = [lognormal_normal_func(i_, *output_lognormal_normal_fit_params_list) 
                         for i_ in x_vals_plot]
#B-type granule graph
# lognormal is defined as (1/(x_*sigma_*np.sqrt(2.*np.pi)))*np.exp( -((np.log(x_)-mu_)/sigma_)**2/2. )
lognormal_normal_B_vals_curve = [lognormal_func(i_, mu_1_lognormal_normal_optimised, sigma_1_lognormal_normal_optimised) for i_ in x_vals_plot] #don't plot this - not adjusted for A and gamma
lognormal_normal_B_vals_curve_to_plot = [i * 100 * gamma_lognormal_normal_optimised for i in lognormal_normal_B_vals_curve] #need this step to adjust curve for the value of A and gamma

#A-type granule graph
# normal is defined as: (  (1/(sigma*np.sqrt(2.*np.pi)))*np.exp( -( (x_ - mu)/sigma )**2/2. )  )
lognormal_normal_A_vals_curve = [normal_func(i_,mu_2_lognormal_normal_optimised, sigma_2_lognormal_normal_optimised) for i_ in x_vals_plot] #don't plot this - not adjusted for A and gamma
lognormal_normal_A_vals_curve_to_plot = [i * 100 * (1-gamma_lognormal_normal_optimised) for i in lognormal_normal_A_vals_curve] #need this step to adjust curve for the value of A and gamma


In [ ]:
# #Plotting everything with the scale the fitting has been performed against
# # Note - the if-else loops are used so that if the fitting failed then the graph is blank and says fitting failed
# # Note - saves the pdf in a folder called output

# #setting up plots
# fig, ax = plt.subplots(4,1,figsize=(15,25))
# fig.suptitle(name, fontsize=20, y=1.01,   fontname="Arial", weight="bold")
# fig.subplots_adjust(top=0.95)
# fig.tight_layout(h_pad=8)
# font = font_manager.FontProperties(family='Arial',
#                                    style='normal', size=11)
# axfont = 11


# #normal-normal graph
# for spine in ['left','right','top','bottom']:
#     ax[0].spines[spine].set_color('k')
# ax[0].set_facecolor('white')
# if fitting_failed_double_normal == "yes":
#     ax[0].text(0.4, 0.5, 'Double normal fitting failed', fontsize = 16)
# else:
#     ax[0].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(),
#             linewidth=2, color='lightgrey',label='Data' )
#     ax[0].plot(x_vals_plot, double_normal_B_vals_curve_to_plot,
#             linewidth=2, color='#CC79A7',label='Normal - B granules', linestyle='dashed' )
#     ax[0].plot(x_vals_plot, double_normal_A_vals_curve_to_plot,
#             linewidth=2, color='#d95f02ff',label='Normal - A granules', linestyle='dashed' )
#     ax[0].plot(x_vals_plot, double_normal_vals_curve,
#             linewidth=2, color='#009e73ff',label='Normal - Normal'  )
#     ax[0].plot(x_vals_plot, double_normal_vals_curve,
#                linewidth=2, color='w', alpha=0, label='Total uncertainty = %.2f\nStandard error of regression =%.5f' %
#            (double_normal_uncertainity, double_normal_standard_error_of_regression))
#     leg_00 = ax[0].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
#     leg_00.get_frame().set_edgecolor('k')

# ax[0].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
# ax[0].xaxis.set_major_locator(plt.MaxNLocator(6))
# ax[0].set_ylabel("Volume (%)", fontsize=11, fontname="Arial")
# ax[0].tick_params(axis='both', which='major', labelsize=11)

# for item in ([ax[0].xaxis.label, ax[0].yaxis.label] +
#              ax[0].get_xticklabels() + ax[0].get_yticklabels()):
#     item.set_fontsize(axfont)

# ax[0].set_title('Normal - Normal (N-N)', fontsize=16, fontname="Arial", weight="bold")

# for tick in ax[0].get_xticklabels():
#     tick.set_fontname("Arial")
# for tick in ax[0].get_yticklabels():
#     tick.set_fontname("Arial")
# ax[0].xaxis.label.set_color('black')
# ax[0].yaxis.label.set_color('black')
# ax[0].tick_params(axis='both', which='major', colors='black')
# ax[0].set_ylim(bottom=0)
# ax[0].set_xlim(left=0)
# ax[0].set_xlim(0,max_x_value)
# ax[0].spines['top'].set_color('white')
# ax[0].spines['right'].set_color('white')

    
# #lognormal-lognormal graph
# for spine in ['left','right','top','bottom']:
#     ax[1].spines[spine].set_color('k')
# ax[1].set_facecolor('white')
# if fitting_failed_double_normal == "yes":
#     ax[1].text(0.4, 0.5, 'Lognormal-normal fitting failed', fontsize = 16)
# else:
#     ax[1].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(),
#             linewidth=2, color='lightgrey',label='Data' )
#     ax[1].plot(x_vals_plot, double_lognormal_B_vals_curve_to_plot,
#             linewidth=2, color='#CC79A7',label='Log-normal - B granules', linestyle='dashed' )
#     ax[1].plot(x_vals_plot, double_lognormal_A_vals_curve_to_plot,
#             linewidth=2, color='#d95f02ff',label='Log-normal - A granules', linestyle='dashed' )
#     ax[1].plot(x_vals_plot, double_lognormal_vals_curve,
#             linewidth=2, color='#009e73ff',label='Log-normal - Log-normal'  )
#     ax[1].plot(x_vals_plot, double_lognormal_vals_curve,
#                linewidth=2, color='w', alpha=0, label='Total uncertainty = %.2f\nStandard error of regression =%.5f' %
#            (double_lognormal_uncertainity, double_lognormal_standard_error_of_regression))
#     leg_00 = ax[1].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
#     leg_00.get_frame().set_edgecolor('k')

# ax[1].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
# ax[1].xaxis.set_major_locator(plt.MaxNLocator(6))
# ax[1].set_ylabel("Volume (%)", fontsize=11, fontname="Arial")
# ax[1].tick_params(axis='both', which='major', labelsize=11)

# for item in ([ax[1].xaxis.label, ax[1].yaxis.label] +
#              ax[1].get_xticklabels() + ax[1].get_yticklabels()):
#     item.set_fontsize(axfont)

# ax[1].set_title('Log-normal - Log-normal (L-L)', fontsize=16, fontname="Arial", weight="bold")

# for tick in ax[1].get_xticklabels():
#     tick.set_fontname("Arial")
# for tick in ax[1].get_yticklabels():
#     tick.set_fontname("Arial")
# ax[1].xaxis.label.set_color('black')
# ax[1].yaxis.label.set_color('black')
# ax[1].tick_params(axis='both', which='major', colors='black')
# ax[1].set_ylim(bottom=0)
# ax[1].set_xlim(left=0)
# ax[1].set_xlim(0,max_x_value)
# ax[1].spines['top'].set_color('white')
# ax[1].spines['right'].set_color('white')

    
# #lognormal-normal graph
# for spine in ['left','right','top','bottom']:
#     ax[2].spines[spine].set_color('k')
# ax[2].set_facecolor('white')
# if fitting_failed_double_normal == "yes":
#     ax[2].text(0.4, 0.5, 'Double lognormal fitting failed', fontsize = 16)
# else:
#     ax[2].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(),
#             linewidth=2, color='lightgrey',label='Data' )
#     ax[2].plot(x_vals_plot, lognormal_normal_B_vals_curve_to_plot,
#             linewidth=2, color='#CC79A7',label='Log-normal - B granules', linestyle='dashed' )
#     ax[2].plot(x_vals_plot, lognormal_normal_A_vals_curve_to_plot,
#             linewidth=2, color='#d95f02ff',label='Normal - A granules', linestyle='dashed' )
#     ax[2].plot(x_vals_plot, lognormal_normal_vals_curve,
#             linewidth=2, color='#009e73ff',label='Log-normal - Normal'  )
#     ax[2].plot(x_vals_plot, lognormal_normal_vals_curve,
#                linewidth=2, color='w', alpha=0, label='Total uncertainty = %.2f\nStandard error of regression =%.5f' %
#            (lognormal_normal_uncertainity, lognormal_normal_standard_error_of_regression))
#     leg_00 = ax[2].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
#     leg_00.get_frame().set_edgecolor('k')

# ax[2].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
# ax[2].xaxis.set_major_locator(plt.MaxNLocator(6))
# ax[2].set_ylabel("Volume (%)", fontsize=11, fontname="Arial")
# ax[2].tick_params(axis='both', which='major', labelsize=11)

# for item in ([ax[2].xaxis.label, ax[2].yaxis.label] +
#              ax[2].get_xticklabels() + ax[2].get_yticklabels()):
#     item.set_fontsize(axfont)

# ax[2].set_title('Log-normal - Normal (L-N)', fontsize=16, fontname="Arial", weight="bold")

# for tick in ax[2].get_xticklabels():
#     tick.set_fontname("Arial")
# for tick in ax[2].get_yticklabels():
#     tick.set_fontname("Arial")
# ax[2].xaxis.label.set_color('black')
# ax[2].yaxis.label.set_color('black')
# ax[2].tick_params(axis='both', which='major', colors='black')
# ax[2].set_ylim(bottom=0)
# ax[2].set_xlim(left=0)
# ax[2].set_xlim(0,max_x_value)
# ax[2].spines['top'].set_color('white')
# ax[2].spines['right'].set_color('white')


# #comparing all graphs
# for spine in ['left','right','top','bottom']:
#     ax[3].spines[spine].set_color('k')
# ax[3].set_facecolor('white')
# ax[3].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(),
#             linewidth=2, color='lightgrey',label='Data' )
# if fitting_failed_double_normal == "yes":
#     pass
# else:
#     ax[3].plot(x_vals_plot, double_normal_vals_curve, 
#         linewidth=2, color='#1e88e5ff',label='N-N, S =%.5f\n' % (double_normal_standard_error_of_regression))  
# if fitting_failed_double_lognormal == "yes":
#     pass
# else:
#     ax[3].plot(x_vals_plot, double_lognormal_vals_curve,
#         linewidth=2, color='#ffc107ff',label='L-L, S =%.5f\n' % (double_lognormal_standard_error_of_regression))
# if fitting_failed_lognormal_normal == "yes":
#     pass
# else:    
#     ax[3].plot(x_vals_plot, lognormal_normal_vals_curve, 
#         linewidth=2, color='#d81b60ff',label='L-N, S =%.5f\n' % (lognormal_normal_standard_error_of_regression))  
   
#     leg_00 = ax[3].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
#     leg_00.get_frame().set_edgecolor('k')

# ax[3].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
# ax[3].xaxis.set_major_locator(plt.MaxNLocator(6))
# ax[3].set_ylabel("Volume (%)", fontsize=11, fontname="Arial")
# ax[3].tick_params(axis='both', which='major', labelsize=11)

# for item in ([ax[3].xaxis.label, ax[3].yaxis.label] +
#              ax[3].get_xticklabels() + ax[3].get_yticklabels()):
#     item.set_fontsize(axfont)

# ax[3].set_title('All fits', fontsize=16, fontname="Arial", weight="bold")

# for tick in ax[3].get_xticklabels():
#     tick.set_fontname("Arial")
# for tick in ax[3].get_yticklabels():
#     tick.set_fontname("Arial")
# ax[3].xaxis.label.set_color('black')
# ax[3].yaxis.label.set_color('black')
# ax[3].tick_params(axis='both', which='major', colors='black')
# ax[3].set_ylim(bottom=0)
# ax[3].set_xlim(left=0)
# ax[3].set_xlim(0,max_x_value)
# ax[3].spines['top'].set_color('white')
# ax[3].spines['right'].set_color('white')


# # #Saving plots in a folder called output
# # plot_name = (name.strip('\n')+"_output.pdf")
# # fig.savefig('output/'+name+".pdf", bbox_inches = 'tight', dpi=300)

In [ ]:
#Plotting everything with y adjusted scale
# Combining all plots together in one pdf file
# Note - the if-else loops are used so that if the fitting failed then the graph is blank and says fitting failed
# Note - saves the pdf in a folder called output


#the y scale has been adjusted so that the sum of the total height of the peaks = 100%, this is similar to previously published data
#this is a simple transformation and does not change the fitting in any way
#to do this we divide the Diff_volume_density by the sum of the Diff_volume_density and * 100 to make a %


#Sums the Diff_volume_density column and convert to %
summed = input_data_df['Diff_volume_density'].sum()
input_data_df['Diff_volume_percentage'] = 100*input_data_df['Diff_volume_density']/summed
diff_volume_percentage_input_list = input_data_df['Diff_volume_percentage'].tolist()
conversion_factor = 100/summed


#Adjusting values to plot by dividing by summed and multiplying by 100% (or just * by conversion_factor)

#x values
x_vals_plot = np.arange(0.5,max_x_value,0.1)

    
#Normal - normal
double_normal_vals_curve_adjusted_y = [i_ * conversion_factor for i_ in double_normal_vals_curve]
double_normal_B_vals_curve_adjusted_y = [i * conversion_factor for i in double_normal_B_vals_curve_to_plot] #need this step to adjust curve for the value of A and gamma
double_normal_A_vals_curve_adjusted_y = [i * conversion_factor for i in double_normal_A_vals_curve_to_plot] #need this step to adjust curve for the value of A and gamma


#lognormal-lognormal
double_lognormal_vals_curve_adjusted_y = [i_ * conversion_factor for i_ in double_lognormal_vals_curve]
double_lognormal_B_vals_curve_adjusted_y = [i * conversion_factor for i in double_lognormal_B_vals_curve_to_plot] #need this step to adjust curve for the value of A and gamma
double_lognormal_A_vals_curve_adjusted_y = [i * conversion_factor for i in double_lognormal_A_vals_curve_to_plot] #need this step to adjust curve for the value of A and gamma


#lognormal-normal
lognormal_normal_vals_curve_adjusted_y = [i_ * conversion_factor for i_ in lognormal_normal_vals_curve]
lognormal_normal_B_vals_curve_adjusted_y = [i * conversion_factor for i in lognormal_normal_B_vals_curve_to_plot] #need this step to adjust curve for the value of A and gamma
lognormal_normal_A_vals_curve_adjusted_y = [i * conversion_factor for i in lognormal_normal_A_vals_curve_to_plot] #need this step to adjust curve for the value of A and gamma






#setting up plots
fig, ax = plt.subplots(4,1,figsize=(15,25))
fig.suptitle(name, fontsize=20, y=1.01,   fontname="Arial", weight="bold")
fig.subplots_adjust(top=0.95)
fig.tight_layout(h_pad=8)
font = font_manager.FontProperties(family='Arial',
                                   style='normal', size=11)
axfont = 11


#normal-normal graph
for spine in ['left','right','top','bottom']:
    ax[0].spines[spine].set_color('k')
ax[0].set_facecolor('white')
if fitting_failed_double_normal == "yes":
    ax[0].text(0.4, 0.5, 'Double normal fitting failed', fontsize = 16)
else:
    ax[0].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_percentage'].tolist(),
            linewidth=2, color='lightgrey',label='Data' )
    ax[0].plot(x_vals_plot, double_normal_B_vals_curve_adjusted_y,
            linewidth=2, color='#CC79A7',label='Normal - B granules', linestyle='dashed' )
    ax[0].plot(x_vals_plot, double_normal_A_vals_curve_adjusted_y,
            linewidth=2, color='#d95f02ff',label='Normal - A granules', linestyle='dashed' )
    ax[0].plot(x_vals_plot, double_normal_vals_curve_adjusted_y,
            linewidth=2, color='#009e73ff',label='Normal - Normal'  )
    ax[0].plot(x_vals_plot, double_normal_vals_curve_adjusted_y,
               linewidth=2, color='w', alpha=0, label='Total uncertainty = %.2f\nStandard error of regression =%.5f' %
           (double_normal_uncertainity, double_normal_standard_error_of_regression))
    leg_00 = ax[0].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
    leg_00.get_frame().set_edgecolor('k')

ax[0].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
ax[0].xaxis.set_major_locator(plt.MaxNLocator(6))
ax[0].set_ylabel("Relative Volume (%)", fontsize=11, fontname="Arial")
ax[0].tick_params(axis='both', which='major', labelsize=11)

for item in ([ax[0].xaxis.label, ax[0].yaxis.label] +
             ax[0].get_xticklabels() + ax[0].get_yticklabels()):
    item.set_fontsize(axfont)

ax[0].set_title('Normal - Normal (N-N)', fontsize=16, fontname="Arial", weight="bold")

for tick in ax[0].get_xticklabels():
    tick.set_fontname("Arial")
for tick in ax[0].get_yticklabels():
    tick.set_fontname("Arial")
ax[0].xaxis.label.set_color('black')
ax[0].yaxis.label.set_color('black')
ax[0].tick_params(axis='both', which='major', colors='black')
ax[0].set_ylim(bottom=0)
ax[0].set_xlim(left=0)
ax[0].set_xlim(0,max_x_value)
ax[0].spines['top'].set_color('white')
ax[0].spines['right'].set_color('white')

    
#lognormal-lognormal graph
for spine in ['left','right','top','bottom']:
    ax[1].spines[spine].set_color('k')
ax[1].set_facecolor('white')
if fitting_failed_double_normal == "yes":
    ax[1].text(0.4, 0.5, 'Lognormal-normal fitting failed', fontsize = 16)
else:
    ax[1].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_percentage'].tolist(),
            linewidth=2, color='lightgrey',label='Data' )
    ax[1].plot(x_vals_plot, double_lognormal_B_vals_curve_adjusted_y,
            linewidth=2, color='#CC79A7',label='Log-normal - B granules', linestyle='dashed' )
    ax[1].plot(x_vals_plot, double_lognormal_A_vals_curve_adjusted_y,
            linewidth=2, color='#d95f02ff',label='Log-normal - A granules', linestyle='dashed' )
    ax[1].plot(x_vals_plot, double_lognormal_vals_curve_adjusted_y,
            linewidth=2, color='#009e73ff',label='Log-normal - Log-normal'  )
    ax[1].plot(x_vals_plot, double_lognormal_vals_curve_adjusted_y,
               linewidth=2, color='w', alpha=0, label='Total uncertainty = %.2f\nStandard error of regression =%.5f' %
           (double_lognormal_uncertainity, double_lognormal_standard_error_of_regression))
    leg_00 = ax[1].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
    leg_00.get_frame().set_edgecolor('k')

ax[1].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
ax[1].xaxis.set_major_locator(plt.MaxNLocator(6))
ax[1].set_ylabel("Relative Volume (%)", fontsize=11, fontname="Arial")
ax[1].tick_params(axis='both', which='major', labelsize=11)

for item in ([ax[1].xaxis.label, ax[1].yaxis.label] +
             ax[1].get_xticklabels() + ax[1].get_yticklabels()):
    item.set_fontsize(axfont)

ax[1].set_title('Log-normal - Log-normal (L-L)', fontsize=16, fontname="Arial", weight="bold")

for tick in ax[1].get_xticklabels():
    tick.set_fontname("Arial")
for tick in ax[1].get_yticklabels():
    tick.set_fontname("Arial")
ax[1].xaxis.label.set_color('black')
ax[1].yaxis.label.set_color('black')
ax[1].tick_params(axis='both', which='major', colors='black')
ax[1].set_ylim(bottom=0)
ax[1].set_xlim(left=0)
ax[1].set_xlim(0,max_x_value)
ax[1].spines['top'].set_color('white')
ax[1].spines['right'].set_color('white')

    
#lognormal-normal graph
for spine in ['left','right','top','bottom']:
    ax[2].spines[spine].set_color('k')
ax[2].set_facecolor('white')
if fitting_failed_double_normal == "yes":
    ax[2].text(0.4, 0.5, 'Double lognormal fitting failed', fontsize = 16)
else:
    ax[2].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_percentage'].tolist(),
            linewidth=2, color='lightgrey',label='Data' )
    ax[2].plot(x_vals_plot, lognormal_normal_B_vals_curve_adjusted_y,
            linewidth=2, color='#CC79A7',label='Log-normal - B granules', linestyle='dashed' )
    ax[2].plot(x_vals_plot, lognormal_normal_A_vals_curve_adjusted_y,
            linewidth=2, color='#d95f02ff',label='Normal - A granules', linestyle='dashed' )
    ax[2].plot(x_vals_plot, lognormal_normal_vals_curve_adjusted_y,
            linewidth=2, color='#009e73ff',label='Log-normal - Normal'  )
    ax[2].plot(x_vals_plot, lognormal_normal_vals_curve_adjusted_y,
               linewidth=2, color='w', alpha=0, label='Total uncertainty = %.2f\nStandard error of regression =%.5f' %
           (lognormal_normal_uncertainity, lognormal_normal_standard_error_of_regression))
    leg_00 = ax[2].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
    leg_00.get_frame().set_edgecolor('k')

ax[2].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
ax[2].xaxis.set_major_locator(plt.MaxNLocator(6))
ax[2].set_ylabel("Relative Volume (%)", fontsize=11, fontname="Arial")
ax[2].tick_params(axis='both', which='major', labelsize=11)

for item in ([ax[2].xaxis.label, ax[2].yaxis.label] +
             ax[2].get_xticklabels() + ax[2].get_yticklabels()):
    item.set_fontsize(axfont)

ax[2].set_title('Log-normal - Normal (L-N)', fontsize=16, fontname="Arial", weight="bold")

for tick in ax[2].get_xticklabels():
    tick.set_fontname("Arial")
for tick in ax[2].get_yticklabels():
    tick.set_fontname("Arial")
ax[2].xaxis.label.set_color('black')
ax[2].yaxis.label.set_color('black')
ax[2].tick_params(axis='both', which='major', colors='black')
ax[2].set_ylim(bottom=0)
ax[2].set_xlim(left=0)
ax[2].set_xlim(0,max_x_value)
ax[2].spines['top'].set_color('white')
ax[2].spines['right'].set_color('white')


#comparing all graphs
for spine in ['left','right','top','bottom']:
    ax[3].spines[spine].set_color('k')
ax[3].set_facecolor('white')
ax[3].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_percentage'].tolist(),
            linewidth=2, color='lightgrey',label='Data' )
if fitting_failed_double_normal == "yes":
    pass
else:
    ax[3].plot(x_vals_plot, double_normal_vals_curve_adjusted_y, 
        linewidth=2, color='#1e88e5ff',label='N-N, S =%.5f\n' % (double_normal_standard_error_of_regression))  
if fitting_failed_double_lognormal == "yes":
    pass
else:
    ax[3].plot(x_vals_plot, double_lognormal_vals_curve_adjusted_y,
        linewidth=2, color='#ffc107ff',label='L-L, S =%.5f\n' % (double_lognormal_standard_error_of_regression))
if fitting_failed_lognormal_normal == "yes":
    pass
else:    
    ax[3].plot(x_vals_plot, lognormal_normal_vals_curve_adjusted_y, 
        linewidth=2, color='#d81b60ff',label='L-N, S =%.5f\n' % (lognormal_normal_standard_error_of_regression))  
   
    leg_00 = ax[3].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
    leg_00.get_frame().set_edgecolor('k')

ax[3].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
ax[3].xaxis.set_major_locator(plt.MaxNLocator(6))
ax[3].set_ylabel("Relative Volume (%)", fontsize=11, fontname="Arial")
ax[3].tick_params(axis='both', which='major', labelsize=11)

for item in ([ax[3].xaxis.label, ax[3].yaxis.label] +
             ax[3].get_xticklabels() + ax[3].get_yticklabels()):
    item.set_fontsize(axfont)

ax[3].set_title('All fits', fontsize=16, fontname="Arial", weight="bold")

for tick in ax[3].get_xticklabels():
    tick.set_fontname("Arial")
for tick in ax[3].get_yticklabels():
    tick.set_fontname("Arial")
ax[3].xaxis.label.set_color('black')
ax[3].yaxis.label.set_color('black')
ax[3].tick_params(axis='both', which='major', colors='black')
ax[3].set_ylim(bottom=0)
ax[3].set_xlim(left=0)
ax[3].set_xlim(0,max_x_value)
ax[3].spines['top'].set_color('white')
ax[3].spines['right'].set_color('white')


#Saving plots in a folder called output
plot_name = (name.strip('\n')+"_output.pdf")
fig.savefig('output/PDFs/'+name+".pdf", bbox_inches = 'tight', dpi=300)

In [ ]:
#Getting the fitting parameters from the different fits
Double_normal_fitting_parameters = (gamma_2_double_normal_optimised, gamma_2_double_normal_uncertainty, 
                                    mu_3_double_normal_optimised, mu_3_double_normal_uncertainty, sigma_3_double_normal_optimised, 
                                    sigma_3_double_normal_uncertainty, mu_4_double_normal_optimised, mu_4_double_normal_uncertainty, 
                                    sigma_4_double_normal_optimised, sigma_4_double_normal_uncertainty)
Double_lognormal_fitting_parameters = (gamma_3_double_lognormal_optimised, gamma_3_double_lognormal_uncertainty, mu_5_double_lognormal_optimised, mu_5_double_lognormal_uncertainty, 
                                       sigma_5_double_lognormal_optimised, sigma_5_double_lognormal_uncertainty, mu_6_double_lognormal_optimised, 
                                       mu_6_double_lognormal_uncertainty, sigma_6_double_lognormal_optimised, sigma_6_double_lognormal_uncertainty)
Lognormal_normal_fitting_parameters = (gamma_lognormal_normal_optimised, gamma_lognormal_normal_uncertainty, mu_1_lognormal_normal_optimised, mu_1_lognormal_normal_uncertainty,
                                       sigma_1_lognormal_normal_optimised, sigma_1_lognormal_normal_uncertainty, mu_2_lognormal_normal_optimised, mu_2_lognormal_normal_uncertainty,
                                       sigma_2_lognormal_normal_optimised, sigma_2_lognormal_normal_uncertainty)

#Joining together the sample name and starch related parameters
name_str = (name,) #converts the sample name into a tuple so it is in the same format as the parameters
fitting_parameters = (name_str + Double_normal_fitting_parameters + Double_lognormal_fitting_parameters + Lognormal_normal_fitting_parameters)



#Getting the starch related parameters from the different fits
Double_normal_parameters = (B_granule_diameter_double_normal, A_granule_diameter_double_normal, B_granule_content_double_normal, A_granule_content_double_normal, B_granule_variance_double_normal, A_granule_variance_double_normal, double_normal_uncertainity, double_normal_standard_error_of_regression)
Double_lognormal_parameters = (B_granule_diameter_double_lognormal, A_granule_diameter_double_lognormal, B_granule_content_double_lognormal, A_granule_content_double_lognormal, B_granule_variance_double_lognormal, A_granule_variance_double_lognormal, double_lognormal_uncertainity, double_lognormal_standard_error_of_regression)
Lognormal_normal_parameters = (B_granule_diameter_lognormal_normal, A_granule_diameter_lognormal_normal, B_granule_content_lognormal_normal, A_granule_content_lognormal_normal, B_granule_variance_lognormal_normal, A_granule_variance_lognormal_normal, lognormal_normal_uncertainity, lognormal_normal_standard_error_of_regression)
#Joining together the sample name and starch related parameters
name_str = (name,) #converts the sample name into a tuple so it is in the same format as the parameters
starch_fitting_parameters = (name_str + Double_normal_parameters + Double_lognormal_parameters + Lognormal_normal_parameters)